# Solana Sniper Bot Reverse-Engineering
This notebook contains the complete pipeline for the Solana Sniper Bot Kaggle competition.
It includes Data Generation, Behavioral Analysis, Feature Engineering, Model Training, and Replica Backtesting.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import polars as pl

# Project Root Setup
PROJECT_ROOT = Path(__file__).resolve().parent.parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "raw" / "extracted"

def generate_synthetic_dataset(
    n_deployments: int = 50_000, 
    bot_buy_ratio: float = 0.0032  # ~0.32% realistic target bot rate
):
    """
    Generates realistic synthetic parquet datasets for testing feature pipelines 
    and backtests locally without loading the 40GB live dataset.
    """
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    np.random.seed(42)

    print(f"[-->] Generating synthetic dataset ({n_deployments:,} deployments)...")

    # 1. Generate Timestamps and Base Keys
    start_slot = 300_000_000
    slots = start_slot + np.sort(np.random.choice(100_000, size=n_deployments, replace=True))
    
    # Generate datetime series
    created_ats = pl.datetime_range(
        start=pl.datetime(2026, 1, 1),
        end=pl.datetime(2026, 2, 1),
        interval="1m",
        eager=True
    ).sample(n_deployments, with_replacement=True)

    token_addresses = [f"Token_{i:06d}_PumpFun" for i in range(n_deployments)]
    deployers = [f"Deployer_{np.random.randint(1, 1000):04d}" for _ in range(n_deployments)]

    # Feature Signals (Dev Buy, Socials, Wallet Age)
    dev_buy_sol = np.random.exponential(scale=0.8, size=n_deployments)
    has_socials = np.random.choice([0, 1], size=n_deployments, p=[0.55, 0.45])
    is_jito_bundle = np.random.choice([0, 1], size=n_deployments, p=[0.70, 0.30])
    deployer_wallet_age_days = np.random.exponential(scale=30.0, size=n_deployments)

    # 2. Deployments DataFrame
    df_deployments = pl.DataFrame({
        "token_address": token_addresses,
        "deployer_address": deployers,
        "slot": slots,
        "created_at": created_ats,
        "dev_buy_sol": dev_buy_sol,
        "has_socials": has_socials,
        "is_jito_bundle": is_jito_bundle,
        "deployer_wallet_age_days": deployer_wallet_age_days,
        "ticker": [f"TICK_{i % 500}" for i in range(n_deployments)],
        "name": [f"Token Name {i}" for i in range(n_deployments)],
    })

    # 3. Simulate Target Bot Logic (Deterministic Probability)
    score = (
        2.0 * ((dev_buy_sol >= 0.4) & (dev_buy_sol <= 3.5)).astype(int) +
        2.5 * has_socials +
        1.5 * is_jito_bundle +
        1.0 * (deployer_wallet_age_days >= 7.0).astype(int) -
        3.5
    )
    probs = 1 / (1 + np.exp(-score))
    
    threshold = np.percentile(probs, (1 - bot_buy_ratio) * 100)
    bought_mask = probs >= threshold
    bought_indices = np.where(bought_mask)[0]

    print(f"[+] Simulated Bot Purchases: {len(bought_indices):,} tokens ({len(bought_indices)/n_deployments*100:.2f}%)")

    # 4. Target Bot Trades Parquet
    bot_trades = []
    for idx_np in bought_indices:
        idx = int(idx_np)  # Cast np.int64 to standard Python int for Polars indexing
        token = token_addresses[idx]
        slot = slots[idx]
        ts = created_ats[idx]
        buy_amount = float(np.random.normal(loc=0.15, scale=0.02)) # Fixed entry sizing ~0.15 SOL
        
        bot_trades.append({
            "token_address": token,
            "trader_address": "5brv79eFZ2rGprXNvqgVJBkBptkkw8GJX1XydJyZLyAr",
            "slot": slot, # Zero-block entry
            "timestamp": ts,
            "tx_type": "buy",
            "sol_amount": max(0.05, buy_amount),
            "position_in_block": 1 if is_jito_bundle[idx] else 2
        })

        # Add simulated sell exit (partial / full)
        roi = float(np.random.choice([1.8, -0.9, 0.4, 3.5], p=[0.20, 0.60, 0.10, 0.10]))
        bot_trades.append({
            "token_address": token,
            "trader_address": "5brv79eFZ2rGprXNvqgVJBkBptkkw8GJX1XydJyZLyAr",
            "slot": slot + np.random.randint(10, 200),
            "timestamp": ts,
            "tx_type": "sell",
            "sol_amount": max(0.01, buy_amount * (1 + roi)),
            "position_in_block": 5
        })

    df_bot_trades = pl.DataFrame(bot_trades)

    # 5. Bonding Curve Trades Parquet (For Backtest Outcome Evaluation)
    all_trades = []
    for i in range(min(5000, n_deployments)):
        token = token_addresses[i]
        slot = slots[i]
        ts = created_ats[i]
        for step in range(3):
            all_trades.append({
                "token_address": token,
                "slot": slot + step * 2,
                "timestamp": ts,
                "sol_amount": float(np.random.exponential(scale=0.2)),
                "tx_type": "buy" if step < 2 else "sell"
            })
    df_all_trades = pl.DataFrame(all_trades)

    # 6. Save Parquet Files
    deployments_path = OUTPUT_DIR / "deployments.parquet"
    bot_trades_path = OUTPUT_DIR / "target_bot_trades.parquet"
    all_trades_path = OUTPUT_DIR / "pumpfun_trades.parquet"

    df_deployments.write_parquet(deployments_path)
    df_bot_trades.write_parquet(bot_trades_path)
    df_all_trades.write_parquet(all_trades_path)

    print(f"\n[âœ“] Synthetic Dataset Successfully Created in: {OUTPUT_DIR}")
    print(f"    â€¢ deployments.parquet: {deployments_path.stat().st_size / (1024**2):.2f} MB")
    print(f"    â€¢ target_bot_trades.parquet: {bot_trades_path.stat().st_size / (1024**2):.2f} MB")
    print(f"    â€¢ pumpfun_trades.parquet: {all_trades_path.stat().st_size / (1024**2):.2f} MB")

if __name__ == "__main__":
    generate_synthetic_dataset()
generate_synthetic_dataset()

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

# -----------------------------------------------------------------------------
# Path Setup
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parent.parent
EXTRACTED_DIR = PROJECT_ROOT / "data" / "raw" / "extracted"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Set clean aesthetic for Kaggle media gallery charts
plt.style.use("seaborn-v0_8-darkgrid" if "seaborn-v0_8-darkgrid" in plt.style.available else "default")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10


def run_behavioral_analysis():
    print("============================================================")
    print("  PART 1: TARGET BOT BEHAVIORAL ANALYSIS (5brv...LyAr)")
    print("============================================================\n")

    # 1. Load Parquet Data via Polars LazyFrames
    deployments_lf = pl.scan_parquet(EXTRACTED_DIR / "deployments.parquet")
    bot_trades_lf = pl.scan_parquet(EXTRACTED_DIR / "target_bot_trades.parquet")

    # Join bot trades with deployment records to analyze latency and zero-block entry
    df_joined = (
        bot_trades_lf.filter(pl.col("tx_type") == "buy")
        .join(deployments_lf, on="token_address", how="inner", suffix="_deploy")
        .collect()
    )

    total_bought = len(df_joined)
    print(f"[+] Total Tokens Bought Analyzed: {total_bought:,}")

    # -------------------------------------------------------------------------
    # A. ENTRY SIZE METRICS
    # -------------------------------------------------------------------------
    buy_sizes = df_joined["sol_amount"]
    mean_entry = buy_sizes.mean()
    median_entry = buy_sizes.median()
    std_entry = buy_sizes.std()
    min_entry = buy_sizes.min()
    max_entry = buy_sizes.max()

    print("\n--- ENTRY SIZE STATISTICS (SOL) ---")
    print(f"  â€¢ Mean Entry Size   : {mean_entry:.4f} SOL")
    print(f"  â€¢ Median Entry Size : {median_entry:.4f} SOL")
    print(f"  â€¢ Std Dispersion    : {std_entry:.4f} SOL")
    print(f"  â€¢ Min / Max Sizing  : {min_entry:.4f} / {max_entry:.4f} SOL")

    # -------------------------------------------------------------------------
    # B. LATENCY & ZERO-BLOCK ANALYSIS
    # -------------------------------------------------------------------------
    df_latency = df_joined.with_columns(
        slot_delta=(pl.col("slot") - pl.col("slot_deploy")),
    )

    zero_block_buys = df_latency.filter(pl.col("slot_delta") == 0)
    zero_block_share = (len(zero_block_buys) / total_bought) * 100

    print("\n--- LATENCY & ZERO-BLOCK SHARE ---")
    print(f"  â€¢ Zero-Block Entries (slot_delta == 0) : {len(zero_block_buys):,} ({zero_block_share:.2f}%)")
    print(f"  â€¢ Mean Slot Latency                     : {df_latency['slot_delta'].mean():.2f} slots")

    # -------------------------------------------------------------------------
    # C. P&L & WIN/LOSS PERFORMANCE
    # -------------------------------------------------------------------------
    df_all_bot = bot_trades_lf.collect()
    
    # Calculate per-token P&L by summing buy and sell SOL amounts
    df_pnl = (
        df_all_bot.group_by("token_address")
        .agg(
            total_spent=pl.col("sol_amount").filter(pl.col("tx_type") == "buy").sum(),
            total_received=pl.col("sol_amount").filter(pl.col("tx_type") == "sell").sum(),
            tx_count=pl.len(),
        )
        .with_columns(
            net_pnl=pl.col("total_received") - pl.col("total_spent"),
            roi=(pl.col("total_received") - pl.col("total_spent")) / (pl.col("total_spent") + 1e-6)
        )
    )

    wins = df_pnl.filter(pl.col("net_pnl") > 0)
    losses = df_pnl.filter(pl.col("net_pnl") <= 0)

    hit_rate = (len(wins) / len(df_pnl)) * 100 if len(df_pnl) > 0 else 0.0
    avg_win = wins["net_pnl"].mean() if len(wins) > 0 else 0.0
    avg_loss = losses["net_pnl"].mean() if len(losses) > 0 else 0.0
    total_pnl = df_pnl["net_pnl"].sum()

    print("\n--- P&L & PERFORMANCE SUMMARY ---")
    print(f"  â€¢ Overall Hit Rate      : {hit_rate:.2f}%")
    print(f"  â€¢ Total Net P&L         : {total_pnl:.2f} SOL")
    print(f"  â€¢ Average Win           : +{avg_win:.4f} SOL")
    print(f"  â€¢ Average Loss          : {avg_loss:.4f} SOL")
    print(f"  â€¢ Profit Factor (Win/Loss): {abs(avg_win / avg_loss):.2f}x" if avg_loss != 0 else "N/A")

    # -------------------------------------------------------------------------
    # D. GENERATE KAGGLE MEDIA GALLERY CHARTS
    # -------------------------------------------------------------------------
    print("\n[-->] Generating Media Gallery plots...")

    # Plot 1: Entry Sizing & Latency Distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.histplot(buy_sizes.to_numpy(), ax=axes[0], kde=True, color="#2b5c8f", bins=30)
    axes[0].axvline(mean_entry, color="red", linestyle="--", label=f"Mean: {mean_entry:.2f} SOL")
    axes[0].axvline(median_entry, color="green", linestyle="-.", label=f"Median: {median_entry:.2f} SOL")
    axes[0].set_title("Target Bot Entry Size Distribution (SOL)")
    axes[0].set_xlabel("Entry Size (SOL)")
    axes[0].set_ylabel("Trade Count")
    axes[0].legend()

    # Latency Bar
    sns.countplot(x=df_latency["slot_delta"].to_numpy(), ax=axes[1], palette="crest")
    axes[1].set_title("Entry Latency Relative to Token Deployment (Slots)")
    axes[1].set_xlabel("Slot Delta (0 = Zero Block)")
    axes[1].set_ylabel("Buy Count")

    plt.tight_layout()
    chart1_path = FIGURES_DIR / "part1_entry_sizing_latency.png"
    plt.savefig(chart1_path, dpi=300)
    plt.close()
    print(f"  [âœ“] Saved: {chart1_path.relative_to(PROJECT_ROOT)}")

    # Plot 2: Per-Trade ROI Distribution
    plt.figure(figsize=(9, 5))
    rois = df_pnl["roi"].to_numpy() * 100
    sns.histplot(rois, bins=40, kde=True, color="#107c41")
    plt.axvline(0, color="black", linewidth=1.2, linestyle="--")
    plt.title(f"Target Bot Trade ROI Distribution (Hit Rate: {hit_rate:.1f}%)")
    plt.xlabel("Return on Investment (ROI %)")
    plt.ylabel("Frequency")
    
    chart2_path = FIGURES_DIR / "part1_pnl_distribution.png"
    plt.tight_layout()
    plt.savefig(chart2_path, dpi=300)
    plt.close()
    print(f"  [âœ“] Saved: {chart2_path.relative_to(PROJECT_ROOT)}")

    print("\n[âœ“] Part 1 Behavioral Analysis Execution Complete!")


if __name__ == "__main__":
    run_behavioral_analysis()
run_behavioral_analysis()

In [ ]:
import os
from pathlib import Path
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
import shap
from sklearn.metrics import average_precision_score, precision_recall_curve, auc

# -----------------------------------------------------------------------------
# Path Setup
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parent.parent
EXTRACTED_DIR = PROJECT_ROOT / "data" / "raw" / "extracted"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def run_feature_engineering_and_modeling():
    print("============================================================")
    print("  PART 2: FEATURE REVERSE-ENGINEERING & MODEL TRAINING")
    print("============================================================\n")

    # 1. Load Parquet Data via Polars
    deployments_lf = pl.scan_parquet(EXTRACTED_DIR / "deployments.parquet")
    bot_trades_lf = pl.scan_parquet(EXTRACTED_DIR / "target_bot_trades.parquet")

    # Extract target bot bought token addresses
    bot_buys = (
        bot_trades_lf.filter(pl.col("tx_type") == "buy")
        .select("token_address")
        .unique()
        .collect()
    )
    bot_bought_set = set(bot_buys["token_address"].to_list())

    print("[-->] Engineering features under strict t_decision truncation...")

    # 2. Polars Feature Pipeline (STRICT t_decision GUARANTEE)
    # Calculate deployer prior deploys ONLY using window before token creation
    df = (
        deployments_lf
        .sort(["deployer_address", "created_at"])
        .with_columns(
            deployer_past_deploys_count=pl.col("token_address")
            .cum_count()
            .over("deployer_address") - 1,
            target_bot_bought=pl.col("token_address").is_in(list(bot_bought_set)).cast(pl.Int32)
        )
        .with_columns(
            # Time of Day Features
            hour_of_day=pl.col("created_at").dt.hour(),
            day_of_week=pl.col("created_at").dt.weekday(),
            sin_hour=np.sin(2 * np.pi * pl.col("created_at").dt.hour() / 24),
            cos_hour=np.cos(2 * np.pi * pl.col("created_at").dt.hour() / 24),
            # Interaction Features
            dev_buy_to_age_ratio=pl.col("dev_buy_sol") / (pl.col("deployer_wallet_age_days") + 1e-4),
            social_x_jito=pl.col("has_socials") * pl.col("is_jito_bundle")
        )
        .collect()
    )

    print(f"[+] Total Processed Records: {len(df):,}")
    print(f"[+] Positive Target Cases ('bot_bought'): {df['target_bot_bought'].sum():,} ({df['target_bot_bought'].mean()*100:.2f}%)")

    # 3. Train / Test Split (Chronological 75% / 25%)
    df_sorted = df.sort("created_at")
    split_idx = int(len(df_sorted) * 0.75)

    feature_cols = [
        "dev_buy_sol",
        "has_socials",
        "is_jito_bundle",
        "deployer_wallet_age_days",
        "deployer_past_deploys_count",
        "sin_hour",
        "cos_hour",
        "dev_buy_to_age_ratio",
        "social_x_jito"
    ]

    train_df = df_sorted[:split_idx]
    test_df = df_sorted[split_idx:]

    X_train = train_df[feature_cols].to_pandas()
    y_train = train_df["target_bot_bought"].to_numpy()

    X_test = test_df[feature_cols].to_pandas()
    y_test = test_df["target_bot_bought"].to_numpy()

    print(f"\n--- DATASET SPLIT SUMMARY ---")
    print(f"  â€¢ Train Set : {len(X_train):,} samples ({y_train.sum()} positive)")
    print(f"  â€¢ Test Set  : {len(X_test):,} samples ({y_test.sum()} positive)")

    # 4. LightGBM Model Training (Tuned for Severe Class Imbalance)
    pos_weight = (len(y_train) - y_train.sum()) / (y_train.sum() + 1e-5)
    
    train_data = lgb.Dataset(X_train, label=y_train)
    test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

    params = {
        "objective": "binary",
        "metric": "average_precision",
        "boosting_type": "gbdt",
        "learning_rate": 0.05,
        "num_leaves": 31,
        "scale_pos_weight": min(pos_weight, 50.0), # Prevent extreme gradient swings
        "verbose": -1,
        "random_state": 42
    }

    print("\n[-->] Training LightGBM Classifier...")
    model = lgb.train(
        params,
        train_data,
        num_boost_round=250,
        valid_sets=[test_data],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )

    # 5. Out-of-Sample Evaluation
    y_probs = model.predict(X_test, num_iteration=model.best_iteration)
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_probs)
    pr_auc_score = auc(recall_vals, precision_vals)

    print("\n--- MODEL PERFORMANCE (HELD-OUT TEST SPLIT) ---")
    print(f"  â€¢ PR-AUC Score                : {pr_auc_score:.4f}")
    print(f"  â€¢ Average Precision (AP)      : {average_precision_score(y_test, y_probs):.4f}")

    # 6. SHAP Interpretability
    print("\n[-->] Computing SHAP Values for Model Interpretability...")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    # Extract feature importance
    mean_shap = np.abs(shap_values[1] if isinstance(shap_values, list) else shap_values).mean(axis=0)
    importance_df = pl.DataFrame({
        "feature": feature_cols,
        "importance": mean_shap
    }).sort("importance", descending=True)

    print("\n--- TOP REVERSE-ENGINEERED FEATURES (SHAP IMPORTANCE) ---")
    for row in importance_df.iter_rows(named=True):
        print(f"  â€¢ {row['feature']:<30} : {row['importance']:.4f}")

    # 7. Generate Media Gallery Feature Importance Chart
    plt.figure(figsize=(10, 6))
    sns.barplot(
        x=importance_df["importance"].to_numpy(),
        y=importance_df["feature"].to_numpy(),
        hue=importance_df["feature"].to_numpy(),
        palette="viridis",
        legend=False
    )
    plt.title("Top-10 Reverse-Engineered Feature Importances (SHAP Values)")
    plt.xlabel("Mean |SHAP Value| (Impact on Model Output)")
    plt.tight_layout()
    chart_path = FIGURES_DIR / "part2_feature_importance.png"
    plt.savefig(chart_path, dpi=300)
    plt.close()
    print(f"\n  [âœ“] Saved: {chart_path.relative_to(PROJECT_ROOT)}")

    # Save test predictions for Part 3 Backtest Engine
    test_df_out = test_df.with_columns(pred_prob=pl.Series(y_probs))
    test_df_out.write_parquet(PROCESSED_DIR / "test_predictions.parquet")
    print(f"  [âœ“] Saved Test Predictions to: {PROCESSED_DIR / 'test_predictions.parquet'}")

    print("\n[âœ“] Part 2 Feature Engineering & Model Training Complete!")


if __name__ == "__main__":
    run_feature_engineering_and_modeling()
run_feature_engineering_and_modeling()

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

# -----------------------------------------------------------------------------
# Path Setup
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
EXTRACTED_DIR = DATA_DIR / "raw" / "extracted"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Plotting aesthetics
plt.style.use("seaborn-v0_8-darkgrid" if "seaborn-v0_8-darkgrid" in plt.style.available else "default")


def run_replica_backtest():
    print("============================================================")
    print("  PART 3: REPLICA STRATEGY & BACKTEST ENGINE")
    print("============================================================\n")

    # 1. Load Processed Test Predictions and Trades
    test_preds_path = PROCESSED_DIR / "test_predictions.parquet"
    bot_trades_path = EXTRACTED_DIR / "target_bot_trades.parquet"

    if not test_preds_path.exists():
        raise FileNotFoundError("Missing test_predictions.parquet. Run part2_feature_engineering.py first.")

    df_test = pl.read_parquet(test_preds_path)
    df_bot_trades = pl.read_parquet(bot_trades_path)

    # Compute target bot ground-truth outcomes on test split
    bot_outcomes = (
        df_bot_trades.group_by("token_address")
        .agg(
            spent=pl.col("sol_amount").filter(pl.col("tx_type") == "buy").sum(),
            received=pl.col("sol_amount").filter(pl.col("tx_type") == "sell").sum(),
        )
        .with_columns(
            net_pnl=pl.col("received") - pl.col("spent"),
            roi=(pl.col("received") - pl.col("spent")) / (pl.col("spent") + 1e-6)
        )
    )

    # Join predictions with ground-truth trade outcomes
    df_eval = df_test.join(bot_outcomes, on="token_address", how="left").fill_null(0.0)

    # -------------------------------------------------------------------------
    # 2. REPLICA STRATEGY DEFINITION & ENTRY SCORING
    # -------------------------------------------------------------------------
    # Operating point threshold (e.g. Prob >= 0.70)
    SCORE_THRESHOLD = 0.70
    FIXED_ENTRY_SOL = 0.15 # Fixed sizing matching target bot

    df_eval = df_eval.with_columns(
        replica_signal=(pl.col("pred_prob") >= SCORE_THRESHOLD).cast(pl.Int32)
    )

    # -------------------------------------------------------------------------
    # 3. BACKTEST SIMULATION WITH SLOT-DELAY SENSITIVITY
    # -------------------------------------------------------------------------
    def simulate_strategy(df_data, slot_delay=0):
        """
        Simulates entry execution with slippage penalties for 1-2 slot delays.
        Solana zero-block price impact increases rapidly with delayed slots.
        """
        entries = df_data.filter(pl.col("replica_signal") == 1)
        if len(entries) == 0:
            return {"Delay": slot_delay, "Trades": 0, "ROI": 0.0, "Hit_Rate": 0.0, "P&L": 0.0, "Max_DD": 0.0}

        # Slippage penalty per slot delay (15% per delayed slot)
        slippage_factor = 1.0 + (0.15 * slot_delay)
        
        # Calculate adjusted ROI and net P&L
        base_rois = entries["roi"].to_numpy()
        adjusted_rois = ((1.0 + base_rois) / slippage_factor) - 1.0
        pnl_sol = adjusted_rois * FIXED_ENTRY_SOL

        cum_pnl = np.cumsum(pnl_sol)
        cum_max = np.maximum.accumulate(cum_pnl)
        drawdowns = cum_pnl - cum_max
        max_dd = drawdowns.min() if len(drawdowns) > 0 else 0.0

        hit_rate = (adjusted_rois > 0).mean() * 100
        total_pnl = pnl_sol.sum()
        avg_roi = adjusted_rois.mean() * 100

        return {
            "Slot Delay": f"{slot_delay} Slot(s)",
            "Trades": len(entries),
            "Hit Rate": f"{hit_rate:.2f}%",
            "Avg ROI": f"{avg_roi:.2f}%",
            "Total P&L": f"{total_pnl:.2f} SOL",
            "Max Drawdown": f"{max_dd:.2f} SOL",
            "pnl_series": cum_pnl
        }

    # Run simulations across 0, 1, and 2 slot delays
    res_delay_0 = simulate_strategy(df_eval, slot_delay=0)
    res_delay_1 = simulate_strategy(df_eval, slot_delay=1)
    res_delay_2 = simulate_strategy(df_eval, slot_delay=2)

    # -------------------------------------------------------------------------
    # 4. HEAD-TO-HEAD COMPARISON MATRIX VS TARGET BOT
    # -------------------------------------------------------------------------
    bot_entries = df_eval.filter(pl.col("target_bot_bought") == 1)
    replica_entries = df_eval.filter(pl.col("replica_signal") == 1)

    # Overlap Metrics (Precision / Recall of Replica vs Bot)
    overlap_count = df_eval.filter(
        (pl.col("target_bot_bought") == 1) & (pl.col("replica_signal") == 1)
    ).height

    replica_precision = (overlap_count / len(replica_entries) * 100) if len(replica_entries) > 0 else 0.0
    replica_recall = (overlap_count / len(bot_entries) * 100) if len(bot_entries) > 0 else 0.0

    print("--- REPLICA OVERLAP WITH TARGET BOT ---")
    print(f"  â€¢ Replica Total Entries  : {len(replica_entries):,}")
    print(f"  â€¢ Target Bot Entries     : {len(bot_entries):,}")
    print(f"  â€¢ Overlapping Entries    : {overlap_count:,}")
    print(f"  â€¢ Replica Precision      : {replica_precision:.2f}% (Share of Replica buys taken by Bot)")
    print(f"  â€¢ Replica Recall         : {replica_recall:.2f}% (Share of Bot buys captured by Replica)")

    print("\n--- SLOT DELAY SENSITIVITY BACKTEST RESULTS ---")
    bt_summary = pl.DataFrame([
        {k: v for k, v in res_delay_0.items() if k != "pnl_series"},
        {k: v for k, v in res_delay_1.items() if k != "pnl_series"},
        {k: v for k, v in res_delay_2.items() if k != "pnl_series"},
    ])
    print(bt_summary)

    # -------------------------------------------------------------------------
    # 5. EQUITY CURVE COMPARISON CHART
    # -------------------------------------------------------------------------
    plt.figure(figsize=(12, 6))
    if len(res_delay_0.get("pnl_series", [])) > 0:
        plt.plot(res_delay_0["pnl_series"], label="Replica Strategy (0 Slot Delay / Zero Block)", color="#107c41", linewidth=2)
        plt.plot(res_delay_1["pnl_series"], label="Replica Strategy (1 Slot Delay)", color="#d97706", linestyle="--")
        plt.plot(res_delay_2["pnl_series"], label="Replica Strategy (2 Slot Delay)", color="#dc2626", linestyle=":")

    plt.title("Backtest Cumulative P&L Equity Curve & Slot Delay Sensitivity")
    plt.xlabel("Trade Number")
    plt.ylabel("Cumulative Net P&L (SOL)")
    plt.legend()
    plt.tight_layout()

    chart_path = FIGURES_DIR / "part3_equity_curve.png"
    plt.savefig(chart_path, dpi=300)
    plt.close()
    print(f"\n  [âœ“] Saved Equity Curve Chart: {chart_path.relative_to(PROJECT_ROOT)}")

    print("\n[âœ“] Part 3 Replica Backtest Execution Complete!")


if __name__ == "__main__":
    run_replica_backtest()
run_replica_backtest()